# User Retention & Engagement Analysis (SQL + Python)
## by Alisha Tajin

#### The goal of this project is to simulate a data set of users and their subscription behavior to study customer churn patterns and prepare for predictive modeling. Specifically, we are generating synthetic data that mimics a small-scale subscription service:
1.) Users Table: Contains demographic information for num_users users.
- Fields include:
    - `user_id` – unique identifier for each user.
    - `age` – random integer between 18–64.
    - `country` – randomly selected from ["US", "Canada", "UK"].
    - `signup_date` – static date (e.g., "2025-01-01").

2.) Subscriptions Table: Contains each user’s subscription information and churn status.
- Fields include:
    - `user_id` – links to the users table.
    - `subscription_type` – randomly assigned as "Basic", "Standard", or "Premium".
    - `monthly_fee` – corresponding price for the subscription tier.
    - `churned` – binary flag (0 = active, 1 = churned) with a 20% churn probability.

This project builds a complete churn prediction pipeline using **SQL and Python** to simulate and analyze a streaming platform’s user data. A SQLite database is created with two relational tables: users (demographics) and subscriptions (plan type, monthly fee, churn status). Using NumPy, 1,000 synthetic users are generated with a 20% simulated churn rate to mimic realistic subscription behavior.

Objectives:
- Ensure one-to-one matching between users and subscriptions.
- Allow the simulation to be rerun safely without duplicating rows, by either clearing tables or using primary keys with INSERT OR REPLACE.
- Prepare the dataset for exploratory data analysis and predictive modeling, such as predicting which users are likely to churn.

Key Considerations:
- IDs must remain unique and consistent across tables.
- Churn column must be numeric for modeling (0/1).
- The dataset is synthetic but structured to reflect realistic subscription behavior.

In [15]:
import os

if os.path.exists("streaming.db"):
    os.remove("streaming.db")

In [2]:
# Import required libraries

import sqlite3              # Allows us to create and interact with a SQL database
import pandas as pd         # Used for data manipulation and analysis
import numpy as np          # Used for generating simulated data
import matplotlib.pyplot as plt  # Used for data visualization

from sklearn.model_selection import train_test_split  # Splits data into training/testing sets
from sklearn.linear_model import LogisticRegression   # Classification model for churn prediction
from sklearn.metrics import classification_report, accuracy_score  # Model evaluation tools

In [3]:
# Create a connection to a SQLite database file
# If the file does not exist, SQLite will automatically create it

conn = sqlite3.connect("streaming.db")
cursor = conn.cursor()

In [4]:
# Create the USERS table
# This stores demographic information about each user

cursor.execute("""
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    age INTEGER,
    country TEXT,
    signup_date TEXT
)
""")

# Create the SUBSCRIPTIONS table
# This stores subscription details and churn status

cursor.execute("""
CREATE TABLE subscriptions (
    user_id INTEGER PRIMARY KEY,
    subscription_type TEXT,
    monthly_fee REAL,
    churned INTEGER
)
""")

conn.commit() # Save changes to the database


In [5]:
# Create the USERS table only if it doesn't already exist
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    age INTEGER,
    country TEXT,
    signup_date TEXT
)
""")

# Create the SUBSCRIPTIONS table only if it doesn't already exist
cursor.execute("""
CREATE TABLE IF NOT EXISTS subscriptions (
    user_id INTEGER PRIMARY KEY,
    subscription_type TEXT,
    monthly_fee REAL,
    churned INTEGER
)
""")

conn.commit()

In [6]:
# Set random seed for reproducibility
np.random.seed(42)

num_users = 1000  # Number of users to simulate

# Generate simulated user data
users_data = [
    (i,
     np.random.randint(18, 65),                 # Random age between 18–64
     np.random.choice(["US", "Canada", "UK"]),  # Random country
     "2025-01-01")                              # Static signup date
    for i in range(1, num_users + 1)
]

# Insert generated users into the database
cursor.executemany("INSERT INTO users VALUES (?, ?, ?, ?)", users_data)

conn.commit()

In [7]:
pd.read_sql_query("SELECT COUNT(*) FROM users", conn) #showing the outcome

,COUNT(*)
0,1000


In [8]:
# Generate subscription data with 20% churn rate

subscriptions_data = [
    (i,
     np.random.choice(["Basic", "Standard", "Premium"]),  # Subscription tier
     np.random.choice([9.99, 14.99, 19.99]),              # Monthly price
     int(np.random.choice([0, 1], p=[0.8, 0.2])))         # 20% probability of churn
    for i in range(1, num_users + 1)
]

# Insert subscription data into database
cursor.executemany("INSERT INTO subscriptions VALUES (?, ?, ?, ?)", subscriptions_data)

conn.commit()

In [9]:
pd.read_sql_query("SELECT COUNT(*) FROM subscriptions", conn) #showing the outcome

,COUNT(*)
0,1000


In [10]:
# Use SQL to join user and subscription data
# This mimics how analysts pull data from relational databases

df_model = pd.read_sql_query("""
SELECT u.age,
       s.monthly_fee,
       s.churned
FROM users u
JOIN subscriptions s
ON u.user_id = s.user_id
""", conn)

# Convert bytes in "churned" to int; leave existing ints as-is
df_model["churned"] = df_model["churned"].apply(
    lambda x: int.from_bytes(x, byteorder="little") if isinstance(x, (bytes, bytearray)) else int(x)
)

In [11]:
df_model.dtypes #checking

age              int64
monthly_fee    float64
churned          int64
dtype: object

In [12]:
df_model["churned"].head() # Ensure all columns are correct numeric types

0    0
1    0
2    0
3    0
4    0
Name: churned, dtype: int64

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Features (X): independent variables used to predict churn
X = df_model[["age", "monthly_fee"]]

# Target (y): what we are trying to predict
y = df_model["churned"]

# We are attempting to predict whether a user will churn based on age and subscription fee.

# Split data into training (80%) and testing (20%)
# random_state ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model Logistic Regression is used for binary classification problems like churn prediction.
model = LogisticRegression()
model.fit(X_train, y_train)

# Generate predictions
predictions = model.predict(X_test)

# Print evaluation metrics
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# We evaluate model performance using accuracy, precision, recall, and F1-score.

Accuracy: 0.79
              precision    recall  f1-score   support

           0       0.79      1.00      0.88       158
           1       0.00      0.00      0.00        42

    accuracy                           0.79       200
   macro avg       0.40      0.50      0.44       200
weighted avg       0.62      0.79      0.70       200



/opt/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
